# Exercise 3 Digitization and Data Analytics - Machine Learning

## Supervised Learning - Bayes Classifier

Content

- Fundamental concepts in interaction with Apache Spark
- Usage of basic algorithms (Bayes Classifier) from the machine library MLLib within Spark

In [15]:
!pip install pyspark

In [16]:
%matplotlib inline
import os


# Adapt the path to point to your iris.scale and iris.csv
scaledIrisDataPath = "iris.scale"
irisDataPath = "iris.csv"


import platform
print('the python version is: ' + platform.python_version())
import pyspark
from pyspark import SparkContext

the python version is: 3.12.13


### Load the Bayes Classifier library:

In [17]:
# $example on$
from pyspark.ml import Pipeline
from pyspark.ml.classification import NaiveBayes, DecisionTreeClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer, VectorAssembler, IndexToString
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# $example off$
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, StructType, StructField, StringType


In [18]:
spark = SparkSession.builder.appName("NaiveBayesClassificationExample").getOrCreate()

Load your data and preprocess it as already done in previous examples. 

In [19]:
# generate DataFrame, use DataFrameReader class to parse csv input
dataframe=spark.read.csv(irisDataPath,
                    schema=StructType([StructField("sepal_length",DoubleType(),True),
                                       StructField("sepal_width",DoubleType(),True),
                                       StructField("petal_length",DoubleType(),True),
                                       StructField("petal_width",DoubleType(),True),
                                       StructField("spezies",StringType(),True)]))
#define a corresponding vector assembler
assembler = VectorAssembler(inputCols=["sepal_length",
                                 "sepal_width",
                                 "petal_length",
                                 "petal_width"],
                      outputCol="features")
dataset = assembler.transform(dataframe)
dataset.printSchema()
dataset.show(15, False)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- spezies: string (nullable = true)
 |-- features: vector (nullable = true)

+------------+-----------+------------+-----------+-----------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|spezies    |features         |
+------------+-----------+------------+-----------+-----------+-----------------+
|5.1         |3.5        |1.4         |0.2        |Iris-setosa|[5.1,3.5,1.4,0.2]|
|4.9         |3.0        |1.4         |0.2        |Iris-setosa|[4.9,3.0,1.4,0.2]|
|4.7         |3.2        |1.3         |0.2        |Iris-setosa|[4.7,3.2,1.3,0.2]|
|4.6         |3.1        |1.5         |0.2        |Iris-setosa|[4.6,3.1,1.5,0.2]|
|5.0         |3.6        |1.4         |0.2        |Iris-setosa|[5.0,3.6,1.4,0.2]|
|5.4         |3.9        |1.7         |0.4        |Iris-setosa|[5.4,3.9,1.7,0.4]|
|4.

#### Deal with Categorical Labels and Variables

#### Exercise:

Define a transformation, that adds indices to the labels in your data (use StringIndexer), and fit it to your data.
Why it is necessary to have a "fit" and a "transform" step to obtain the labels in the indexed manner?

In [20]:
# Convert target into numerical categories
labelIndexer = StringIndexer(inputCol="spezies", outputCol="indexedLabel")

In [21]:
labelIndexerModel = labelIndexer.fit(dataframe)
dataframeLabelIndexed = labelIndexerModel.transform(dataframe)
dataframeLabelIndexed.printSchema()
dataframeLabelIndexed.show(15, False)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- spezies: string (nullable = true)
 |-- indexedLabel: double (nullable = false)

+------------+-----------+------------+-----------+-----------+------------+
|sepal_length|sepal_width|petal_length|petal_width|spezies    |indexedLabel|
+------------+-----------+------------+-----------+-----------+------------+
|5.1         |3.5        |1.4         |0.2        |Iris-setosa|0.0         |
|4.9         |3.0        |1.4         |0.2        |Iris-setosa|0.0         |
|4.7         |3.2        |1.3         |0.2        |Iris-setosa|0.0         |
|4.6         |3.1        |1.5         |0.2        |Iris-setosa|0.0         |
|5.0         |3.6        |1.4         |0.2        |Iris-setosa|0.0         |
|5.4         |3.9        |1.7         |0.4        |Iris-setosa|0.0         |
|4.6         |3.4        |1.4         |0.3 

#### Why fit() and transform()?
`StringIndexer` is an estimator that must first learn the mapping from string labels to numeric indices with `fit()`. After the mapping is learned, `transform()` applies the same encoding consistently to training and test data.

#### Training and Test Data Sets

#### Exercise:
Split your data into training and test data.

In [22]:
# Split dataset randomly into Training and Test sets (30% held out for testing). Set seed for reproducibility
(trainingData, testData) = dataframe.randomSplit([0.7, 0.3], seed = 100)

print(trainingData.count())
print(testData.count())

104
46


#### Exercise:
Use the above defined label indexer to show the indexed test- and training data.

In [23]:
trainDataLabelIndexed = labelIndexerModel.transform(trainingData)
testDataLabelIndexed = labelIndexerModel.transform(testData)
trainDataLabelIndexed.select("spezies", "indexedLabel", "sepal_length", "sepal_width", "petal_length", "petal_width").show(10, False)
testDataLabelIndexed.select("spezies", "indexedLabel", "sepal_length", "sepal_width", "petal_length", "petal_width").show(10, False)

+-----------+------------+------------+-----------+------------+-----------+
|spezies    |indexedLabel|sepal_length|sepal_width|petal_length|petal_width|
+-----------+------------+------------+-----------+------------+-----------+
|Iris-setosa|0.0         |4.3         |3.0        |1.1         |0.1        |
|Iris-setosa|0.0         |4.4         |2.9        |1.4         |0.2        |
|Iris-setosa|0.0         |4.4         |3.0        |1.3         |0.2        |
|Iris-setosa|0.0         |4.5         |2.3        |1.3         |0.3        |
|Iris-setosa|0.0         |4.6         |3.1        |1.5         |0.2        |
|Iris-setosa|0.0         |4.6         |3.2        |1.4         |0.2        |
|Iris-setosa|0.0         |4.6         |3.4        |1.4         |0.3        |
|Iris-setosa|0.0         |4.6         |3.6        |1.0         |0.2        |
|Iris-setosa|0.0         |4.7         |3.2        |1.3         |0.2        |
|Iris-setosa|0.0         |4.8         |3.0        |1.4         |0.3        |

#### Train a Naive Bayes Classifier

#### Exercise:
Define a Naive Bayes classifier for the iris data set. Have a look at the documentation of  NaiveBayes() and its parameters:  
1. https://spark.apache.org/docs/2.2.0/api/python/_modules/pyspark/ml/classification.html  
2. https://spark.apache.org/docs/2.2.0/api/python/pyspark.ml.html?highlight=naivebayes#pyspark.ml.classification.NaiveBayes).

Hint: Pay attention to the labels within your data, you use for the classifier.

In [24]:
nb = NaiveBayes(featuresCol="features", labelCol="indexedLabel", predictionCol="prediction", smoothing=1.0, modelType="multinomial")

#### Exercise:
1. Read the documentation about the pipeline-component (see https://spark.apache.org/docs/latest/ml-pipeline.html#pipeline-components).
2. Define a pipeline, describing the process of indexing and transforming your data, and then training the naive bayes classifier.
3. Train the model.

In [25]:
pipeline = Pipeline(stages=[assembler, labelIndexer, nb])

In [26]:
# Run stages in pipeline and train model
model = pipeline.fit(dataframe)

#### Use the trained model for predictions

In [27]:
# Make predictions.
predictions = model.transform(testData)

In [28]:
# Select example rows to display.
predictions.select("prediction", "indexedLabel", "spezies", "features").show(15, False)

+----------+------------+---------------+-----------------+
|prediction|indexedLabel|spezies        |features         |
+----------+------------+---------------+-----------------+
|0.0       |0.0         |Iris-setosa    |[4.4,3.2,1.3,0.2]|
|0.0       |0.0         |Iris-setosa    |[4.7,3.2,1.6,0.2]|
|0.0       |0.0         |Iris-setosa    |[4.8,3.0,1.4,0.1]|
|0.0       |0.0         |Iris-setosa    |[4.8,3.4,1.6,0.2]|
|0.0       |0.0         |Iris-setosa    |[4.8,3.4,1.9,0.2]|
|0.0       |0.0         |Iris-setosa    |[4.9,3.1,1.5,0.1]|
|0.0       |0.0         |Iris-setosa    |[4.9,3.1,1.5,0.1]|
|0.0       |0.0         |Iris-setosa    |[5.0,3.2,1.2,0.2]|
|0.0       |0.0         |Iris-setosa    |[5.0,3.5,1.3,0.3]|
|0.0       |0.0         |Iris-setosa    |[5.1,3.4,1.5,0.2]|
|0.0       |0.0         |Iris-setosa    |[5.1,3.5,1.4,0.2]|
|0.0       |0.0         |Iris-setosa    |[5.1,3.7,1.5,0.4]|
|0.0       |0.0         |Iris-setosa    |[5.1,3.8,1.5,0.3]|
|0.0       |0.0         |Iris-setosa    

26/06/09 23:20:03 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [29]:
# Show the same with the original labels
converter = IndexToString(inputCol="prediction", outputCol="predictedLabel", labels=labelIndexerModel.labels)
convertedPrediction = converter.transform(predictions)
convertedPrediction.select("prediction", "predictedLabel", "indexedLabel", "spezies", "features").show(15, False)

+----------+---------------+------------+---------------+-----------------+
|prediction|predictedLabel |indexedLabel|spezies        |features         |
+----------+---------------+------------+---------------+-----------------+
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.4,3.2,1.3,0.2]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.7,3.2,1.6,0.2]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.8,3.0,1.4,0.1]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.8,3.4,1.6,0.2]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.8,3.4,1.9,0.2]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.9,3.1,1.5,0.1]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[4.9,3.1,1.5,0.1]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[5.0,3.2,1.2,0.2]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[5.0,3.5,1.3,0.3]|
|0.0       |Iris-setosa    |0.0         |Iris-setosa    |[5.1,3.4,1.5,0.2]|
|0.0       |

#### Evaluate your results

In [30]:
# Select (prediction, true label) and compute test error
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="f1")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedRecall")

accuracy = accuracy_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)
f1_score = f1_evaluator.evaluate(predictions)

print("Accuracy = %g" % accuracy)
print("Test Error = %g " % (1.0 - accuracy))
print("Weighted Precision = %g" % precision)
print("Weighted Recall = %g" % recall)
print("Weighted F1 = %g" % f1_score)

Accuracy = 0.956522
Test Error = 0.0434783 
Weighted Precision = 0.956522
Weighted Recall = 0.956522
Weighted F1 = 0.956522


#### Visualization of Evaluation Metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a comprehensive metrics visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Bar chart of all metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1_score]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

axes[0, 0].bar(metrics, values, color=colors, alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Classification Metrics Comparison')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(values):
    axes[0, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Error breakdown
test_error = 1.0 - accuracy
errors_data = [accuracy, test_error]
axes[0, 1].pie(errors_data, labels=['Correct', 'Incorrect'], autopct='%1.1f%%', 
               colors=['#2ca02c', '#d62728'], startangle=90)
axes[0, 1].set_title('Prediction Accuracy Breakdown')

# Metrics radar-like bar chart (normalized to 0-1)
x_pos = np.arange(len(metrics))
axes[1, 0].barh(x_pos, values, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].set_yticks(x_pos)
axes[1, 0].set_yticklabels(metrics)
axes[1, 0].set_xlabel('Score')
axes[1, 0].set_title('Detailed Metrics Performance')
axes[1, 0].set_xlim([0, 1])
axes[1, 0].grid(axis='x', alpha=0.3)
for i, v in enumerate(values):
    axes[1, 0].text(v + 0.02, i, f'{v:.3f}', va='center', fontweight='bold')

# Confusion matrix visualization
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
confusion_matrix_data = predictions.groupBy("indexedLabel", "prediction").count().collect()
labels = sorted(set([row.indexedLabel for row in confusion_matrix_data] + [row.prediction for row in confusion_matrix_data]))
cm = np.zeros((len(labels), len(labels)))

for row in confusion_matrix_data:
    cm[int(row.indexedLabel), int(row.prediction)] = row['count']

im = axes[1, 1].imshow(cm, cmap='Blues', aspect='auto')
axes[1, 1].set_xlabel('Predicted Label')
axes[1, 1].set_ylabel('True Label')
axes[1, 1].set_title('Confusion Matrix')
axes[1, 1].set_xticks(range(len(labels)))
axes[1, 1].set_yticks(range(len(labels)))
axes[1, 1].set_xticklabels(labels)
axes[1, 1].set_yticklabels(labels)

# Add text annotations to confusion matrix
for i in range(len(labels)):
    for j in range(len(labels)):
        text = axes[1, 1].text(j, i, int(cm[i, j]), ha="center", va="center", color="black", fontweight='bold')

plt.colorbar(im, ax=axes[1, 1], label='Count')
fig.tight_layout()
plt.show()

print("\nConfusion Matrix Summary:")
print("Rows: True Labels, Columns: Predicted Labels")

#### Results and answers

- Using a 70/30 split gives about 105 training rows and 45 test rows for the Iris dataset. That is enough to keep the error in a reasonable interval for this simple, well-separated dataset.
- When you change the metric, you can observe:
  - `accuracy`: overall fraction of correct predictions,
  - `weightedPrecision`: precision averaged across classes,
  - `weightedRecall`: recall averaged across classes,
  - `f1`: harmonic mean of precision and recall.
- For Iris, accuracy, weighted precision, weighted recall, and F1 are often very similar for a good model, which shows that the classifier is balanced across classes.

In [31]:
spark.stop()